# CT-FM

In [1]:
import torch
from lighter_zoo import SegResEncoder
from monai.transforms import (
    Compose, LoadImage, EnsureType, Orientation,
    ScaleIntensityRange, CropForeground
)
from monai.inferers import SlidingWindowInferer

In [ ]:
# Load pre-trained model
model = SegResEncoder.from_pretrained(
    "project-lighter/ct_fm_feature_extractor"
)
model.eval()

In [ ]:
# Preprocessing pipeline
preprocess = Compose([
    LoadImage(ensure_channel_first=True),  # Load image and ensure channel dimension
    EnsureType(),                         # Ensure correct data type
    Orientation(axcodes="SPL"),           # Standardize orientation
    # Scale intensity to [0,1] range, clipping outliers
    ScaleIntensityRange(
        a_min=-1024,    # Min HU value
        a_max=2048,     # Max HU value
        b_min=0,        # Target min
        b_max=1,        # Target max
        clip=True       # Clip values outside range
    ),
    CropForeground()    # Remove background to reduce computation
])


In [ ]:
# Input path
input_path = r"c:\Users\bguet\Desktop\GrandChallenge\HECKTOR 2025 Training Data\Task 1\CHUM-001\CHUM-001.nii.gz"

# Preprocess input
input_tensor = preprocess(input_path)

# Run inference
with torch.no_grad():
    output = model(input_tensor.unsqueeze(0))[-1]

    # Average pooling compressed the feature vector across all patches. If this is not desired, remove this line and 
    # use the output tensor directly which will give you the feature maps in a low-dimensional space.
    avg_output = torch.nn.functional.adaptive_avg_pool3d(output, 1).squeeze()

print("Feature extraction completed")
print(f"Output shape: {avg_output.shape}")


In [ ]:
# Plot distribution of features
import matplotlib.pyplot as plt
_ = plt.hist(avg_output.cpu().numpy(), bins=100)

# SuPreM

In [ ]:
from monai.transforms import EnsureChannelFirstd, Compose, LoadImaged, ScaleIntensityRanged, ToTensord
test_transforms = Compose(
    [
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
        ScaleIntensityRanged(keys=["image"], a_min=-175, a_max=250, b_min=0.0, b_max=1.0, clip=True,),
        ToTensord(keys=["image"]),
    ])

In [4]:
from monai.networks.nets import SegResNet
import torch

checkpoint = r"C:\Users\bilel.guetarni\Downloads\supervised_suprem_segresnet_2100.pth"

model = SegResNet(
                    blocks_down=[1, 2, 2, 4],
                    blocks_up=[1, 1, 1],
                    init_filters=16,
                    in_channels=1,
                    out_channels=25,
                    dropout_prob=0.0,
                    )

store_dict = model.state_dict()
model_dict = torch.load(checkpoint, map_location='cpu')['net']
new_model_dict={}
for key, value in model_dict.items():
    new_key = key.replace('module.', '')
    new_model_dict[key] = value
model_dict = new_model_dict  
amount = 0
for key in model_dict.keys():
    new_key = '.'.join(key.split('.')[1:])
    if new_key in store_dict.keys():
        store_dict[new_key] = model_dict[key]   
        amount += 1
assert amount == len(store_dict), "the model is not loaded successfully"

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [5]:
# Input path
input_path = r"E:\bilel\HECKTOR 2025 Training Data\Task 1\CHUM-001\CHUM-001__CT.nii.gz"

# Preprocess input
input_tensor = test_transforms({"image": input_path})
input_tensor = input_tensor["image"]
print(input_tensor.shape)
input_tensor = input_tensor.unsqueeze(0)
print(input_tensor.shape)

# Run inference
with torch.no_grad():
    output, _ = model.encode(input_tensor)

    # Average pooling compressed the feature vector across all patches. If this is not desired, remove this line and 
    # use the output tensor directly which will give you the feature maps in a low-dimensional space.
    avg_output = torch.nn.functional.adaptive_avg_pool3d(output, 1).squeeze()

print(avg_output)

torch.Size([1, 512, 512, 91])
torch.Size([1, 1, 512, 512, 91])
metatensor([ 0.5493, -0.0431,  0.1973, -0.6456, -0.0174,  0.2793,  0.7150,  0.4072,
        -0.0793,  0.2978,  0.2530, -0.3751,  0.7228,  0.0185, -0.7830, -0.7583,
         0.0465, -0.1159,  0.3581, -0.7301, -0.0117, -0.2247,  0.3391, -0.2077,
        -0.7328, -0.0688, -0.4493,  0.4389,  0.1096, -0.5693,  0.0582, -0.8704,
         0.2288, -0.5033,  0.0621, -0.8095,  0.7334,  0.0352,  0.3729,  0.0624,
         0.4206,  0.6039, -0.5490,  0.5511,  0.5066,  0.2549,  0.8218,  0.2368,
         0.0730,  0.1964, -0.6990,  0.3687,  0.0279,  0.4472,  0.3181,  0.0371,
         0.7811,  0.1431, -0.2258,  0.8426, -0.1485, -0.3884, -0.2374, -0.4660,
         0.4997, -0.1514,  0.2590,  0.2106,  0.6583, -0.4992,  0.3033,  0.1787,
         0.0057, -0.4608,  0.4177, -0.1338, -1.0023,  0.0465, -0.0377,  0.1249,
         0.2125, -0.0620, -0.0872,  0.7833, -0.5877,  0.6305, -1.0298,  0.5951,
        -0.4692,  0.1250, -0.7808,  0.0075, -0.9473, 

# Model Genesis

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ContBatchNorm3d(nn.modules.batchnorm._BatchNorm):
    def _check_input_dim(self, input):

        if input.dim() != 5:
            raise ValueError('expected 5D input (got {}D input)'.format(input.dim()))
        #super(ContBatchNorm3d, self)._check_input_dim(input)

    def forward(self, input):
        self._check_input_dim(input)
        return F.batch_norm(
            input, self.running_mean, self.running_var, self.weight, self.bias,
            True, self.momentum, self.eps)


class LUConv(nn.Module):
    def __init__(self, in_chan, out_chan, act):
        super(LUConv, self).__init__()
        self.conv1 = nn.Conv3d(in_chan, out_chan, kernel_size=3, padding=1)
        self.bn1 = ContBatchNorm3d(out_chan)

        if act == 'relu':
            self.activation = nn.ReLU(out_chan)
        elif act == 'prelu':
            self.activation = nn.PReLU(out_chan)
        elif act == 'elu':
            self.activation = nn.ELU(inplace=True)
        else:
            raise

    def forward(self, x):
        out = self.activation(self.bn1(self.conv1(x)))
        return out


def _make_nConv(in_channel, depth, act, double_chnnel=False):
    if double_chnnel:
        layer1 = LUConv(in_channel, 32 * (2 ** (depth+1)),act)
        layer2 = LUConv(32 * (2 ** (depth+1)), 32 * (2 ** (depth+1)),act)
    else:
        layer1 = LUConv(in_channel, 32*(2**depth),act)
        layer2 = LUConv(32*(2**depth), 32*(2**depth)*2,act)

    return nn.Sequential(layer1,layer2)


# class InputTransition(nn.Module):
#     def __init__(self, outChans, elu):
#         super(InputTransition, self).__init__()
#         self.conv1 = nn.Conv3d(1, 16, kernel_size=5, padding=2)
#         self.bn1 = ContBatchNorm3d(16)
#         self.relu1 = ELUCons(elu, 16)
#
#     def forward(self, x):
#         # do we want a PRELU here as well?
#         out = self.bn1(self.conv1(x))
#         # split input in to 16 channels
#         x16 = torch.cat((x, x, x, x, x, x, x, x,
#                          x, x, x, x, x, x, x, x), 1)
#         out = self.relu1(torch.add(out, x16))
#         return out

class DownTransition(nn.Module):
    def __init__(self, in_channel,depth, act):
        super(DownTransition, self).__init__()
        self.ops = _make_nConv(in_channel, depth,act)
        self.maxpool = nn.MaxPool3d(2)
        self.current_depth = depth

    def forward(self, x):
        if self.current_depth == 3:
            out = self.ops(x)
            out_before_pool = out
        else:
            out_before_pool = self.ops(x)
            out = self.maxpool(out_before_pool)
        return out, out_before_pool

class UpTransition(nn.Module):
    def __init__(self, inChans, outChans, depth,act):
        super(UpTransition, self).__init__()
        self.depth = depth
        self.up_conv = nn.ConvTranspose3d(inChans, outChans, kernel_size=2, stride=2)
        self.ops = _make_nConv(inChans+ outChans//2,depth, act, double_chnnel=True)

    def forward(self, x, skip_x):
        out_up_conv = self.up_conv(x)
        concat = torch.cat((out_up_conv,skip_x),1)
        out = self.ops(concat)
        return out


class OutputTransition(nn.Module):
    def __init__(self, inChans, n_labels):

        super(OutputTransition, self).__init__()
        self.final_conv = nn.Conv3d(inChans, n_labels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.sigmoid(self.final_conv(x))
        return out

class UNet3D(nn.Module):
    # the number of convolutions in each layer corresponds
    # to what is in the actual prototxt, not the intent
    def __init__(self, n_class=1, act='relu'):
        super(UNet3D, self).__init__()

        self.down_tr64 = DownTransition(1,0,act)
        self.down_tr128 = DownTransition(64,1,act)
        self.down_tr256 = DownTransition(128,2,act)
        self.down_tr512 = DownTransition(256,3,act)

        self.up_tr256 = UpTransition(512, 512,2,act)
        self.up_tr128 = UpTransition(256,256, 1,act)
        self.up_tr64 = UpTransition(128,128,0,act)
        self.out_tr = OutputTransition(64, n_class)

    def forward(self, x):
        self.out64, self.skip_out64 = self.down_tr64(x)
        self.out128,self.skip_out128 = self.down_tr128(self.out64)
        self.out256,self.skip_out256 = self.down_tr256(self.out128)
        self.out512, self.skip_out512 = self.down_tr512(self.out256)

        # self.out_up_256 = self.up_tr256(self.out512,self.skip_out256)
        # self.out_up_128 = self.up_tr128(self.out_up_256, self.skip_out128)
        # self.out_up_64 = self.up_tr64(self.out_up_128, self.skip_out64)
        # self.out = self.out_tr(self.out_up_64)

        # return self.out
        return self.out512

In [11]:
model = UNet3D()

#Load pre-trained weights
weight_dir = r"C:\Users\bilel.guetarni\Downloads\Genesis_Chest_CT.pt"
checkpoint = torch.load(weight_dir)
state_dict = checkpoint['state_dict']
unParalled_state_dict = {}
for key in state_dict.keys():
    unParalled_state_dict[key.replace("module.", "")] = state_dict[key]
model.load_state_dict(unParalled_state_dict)

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


<All keys matched successfully>

In [ ]:
from monai.transforms import EnsureChannelFirstd, Compose, LoadImaged, ScaleIntensityRanged, ToTensord

transforms_ = Compose(
    [
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
        ToTensord(keys=["image"]),
    ])

In [18]:
# Input path
input_path = r"E:\bilel\HECKTOR 2025 Training Data\Task 1\CHUM-001\CHUM-001__CT.nii.gz"

# Preprocess input
input_tensor = test_transforms({"image": input_path})
input_tensor = input_tensor["image"]
print(input_tensor.shape)
input_tensor = input_tensor.unsqueeze(0)
print(input_tensor.shape)

# Run inference
with torch.no_grad():
    output = model(input_tensor)

    # Average pooling compressed the feature vector across all patches. If this is not desired, remove this line and 
    # use the output tensor directly which will give you the feature maps in a low-dimensional space.
    avg_output = torch.nn.functional.adaptive_avg_pool3d(output, 1).squeeze()

print(avg_output.shape)

torch.Size([1, 512, 512, 91])
torch.Size([1, 1, 512, 512, 91])
torch.Size([512])
